Ingestão de dados e tabela Bronze

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS delivery;

In [0]:
# dbfs:/FileStore/delivery/

In [0]:
df = (
    spark.read
    .option("header","true")
    .option("inferSchema","true")
    .csv("/Volumes/workspace/delivery/delivery_data/online_food_delivery_dataset.csv")
)

In [0]:
from pyspark.sql.functions import col

# Sanitize column names: replace spaces with underscores
for old_name in df.columns:
    new_name = old_name.replace(" ", "_")
    if old_name != new_name:
        df = df.withColumnRenamed(old_name, new_name)

df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("delivery.bronze_online_delivery")

### **Tatamento de qualidade - Camada Silver**

In [0]:
from pyspark.sql.functions import col

df = spark.table("delivery.bronze_online_delivery")

for c in df.columns:
    df = df.withColumnRenamed(
        c,
        c.strip()
         .replace(" ","_")
         .replace(".","")
         .lower()
    )

In [0]:
df = df.drop("_c13")

Criar Labels

In [0]:
from pyspark.sql.functions import when

df = df.withColumn(
    "label_output",
    when(col("output")=="Yes",1).otherwise(0)
)

In [0]:
df = df.withColumn(
    "label_feedback",
    when(col("feedback").contains("Positive"),1)
    .otherwise(0)
)

In [0]:
display(df.limit(20))

age,gender,marital_status,occupation,monthly_income,educational_qualifications,family_size,customer_type,latitude,longitude,pin_code,output,feedback,label_output,label_feedback
20,Female,Single,Student,No Income,Post Graduate,4,Frequent,12.9766,77.5993,560001,Yes,Positive,1,1
24,Female,Single,Student,Below Rs.10000,Graduate,3,Regular,12.977,77.5773,560009,Yes,Positive,1,1
22,Male,Single,Student,Below Rs.10000,Post Graduate,3,Regular,12.9551,77.6593,560017,Yes,Negative,1,0
22,Female,Single,Student,No Income,Graduate,6,Frequent,12.9473,77.5616,560019,Yes,Positive,1,1
22,Male,Single,Student,Below Rs.10000,Post Graduate,4,Frequent,12.985,77.5533,560010,Yes,Positive,1,1
27,Female,Married,Employee,More than 50000,Post Graduate,2,Regular,12.9299,77.6848,560103,Yes,Positive,1,1
22,Male,Single,Student,No Income,Graduate,3,Regular,12.977,77.5773,560009,Yes,Positive,1,1
24,Female,Single,Student,No Income,Post Graduate,3,Regular,12.9828,77.6131,560042,Yes,Positive,1,1
23,Female,Single,Student,No Income,Post Graduate,2,Regular,12.9766,77.5993,560001,Yes,Positive,1,1
23,Female,Single,Student,No Income,Post Graduate,4,Frequent,12.9854,77.7081,560048,Yes,Positive,1,1


**ENGENHARIA DE FEATURES**

Faixa etária

In [0]:
df = df.withColumn(
    "age_group",
    when(col("age") < 25,"young")
    .when(col("age") < 30,"adult")
    .otherwise("senior")
)

Tamanho da família

In [0]:
df = df.withColumn(
    "big_family",
    when(col("family_size") >= 5,1)
    .otherwise(0)
)

Cliente recorrente

In [0]:
df = df.withColumn(
    "is_loyal",
    when(col("customer_type")=="Frequent",1)
    .otherwise(0)
)

In [0]:
display(df.limit(20))

age,gender,marital_status,occupation,monthly_income,educational_qualifications,family_size,customer_type,latitude,longitude,pin_code,output,feedback,label_output,label_feedback,age_group,big_family,is_loyal
20,Female,Single,Student,No Income,Post Graduate,4,Frequent,12.9766,77.5993,560001,Yes,Positive,1,1,young,0,1
24,Female,Single,Student,Below Rs.10000,Graduate,3,Regular,12.977,77.5773,560009,Yes,Positive,1,1,young,0,0
22,Male,Single,Student,Below Rs.10000,Post Graduate,3,Regular,12.9551,77.6593,560017,Yes,Negative,1,0,young,0,0
22,Female,Single,Student,No Income,Graduate,6,Frequent,12.9473,77.5616,560019,Yes,Positive,1,1,young,1,1
22,Male,Single,Student,Below Rs.10000,Post Graduate,4,Frequent,12.985,77.5533,560010,Yes,Positive,1,1,young,0,1
27,Female,Married,Employee,More than 50000,Post Graduate,2,Regular,12.9299,77.6848,560103,Yes,Positive,1,1,adult,0,0
22,Male,Single,Student,No Income,Graduate,3,Regular,12.977,77.5773,560009,Yes,Positive,1,1,young,0,0
24,Female,Single,Student,No Income,Post Graduate,3,Regular,12.9828,77.6131,560042,Yes,Positive,1,1,young,0,0
23,Female,Single,Student,No Income,Post Graduate,2,Regular,12.9766,77.5993,560001,Yes,Positive,1,1,young,0,0
23,Female,Single,Student,No Income,Post Graduate,4,Frequent,12.9854,77.7081,560048,Yes,Positive,1,1,young,0,1


Cluster Geográfico

In [0]:
from pyspark.ml.clustering import KMeans

In [0]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.clustering import KMeans

# Assemble geo features into a single vector column
assembler = VectorAssembler(
    inputCols=["latitude", "longitude", "pin_code"],
    outputCol="geo_features"
)
df = assembler.transform(df)

# Train KMeans clustering model (k=5 as default)
kmeans = KMeans(k=5, seed=42, featuresCol="geo_features", predictionCol="region_cluster")
model = kmeans.fit(df)

# Add cluster prediction column
df = model.transform(df)

# Drop the intermediate vector column
df = df.drop("geo_features")

In [0]:
features = [
    "age",
    "family_size",
    "latitude",
    "longitude",
    "region_cluster"
]

In [0]:
categorical_columns = [
    "gender",
    "marital_status",
    "occupation",
    "monthly_income",
    "educational_qualifications",
    "customer_type",
    "age_group"
]

In [0]:
numeric_cols = [
    "age",
    "family_size",
    "big_family",
    "is_loyal"
]

In [0]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml import Pipeline

# Index and one-hot encode categorical columns
indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in categorical_columns
]

encoder = OneHotEncoder(
    inputCols=[f"{c}_idx" for c in categorical_columns],
    outputCols=[f"{c}_ohe" for c in categorical_columns]
)

# Assemble all features into a single vector column

assembler = VectorAssembler(
    inputCols=
        numeric_cols +
        [f"{c}_ohe" for c in categorical_columns],
    outputCol="features"
)

# Drop any previously created intermediate columns
for c in df.columns:
    if c.endswith("_idx") or c.endswith("_ohe") or c.endswith("_vec") or c == "features":
        df = df.drop(c)

# Build and fit the preprocessing pipeline

pipeline_features = Pipeline(
    stages=
        indexers +
        [encoder, assembler]
)
feature_model = pipeline_features.fit(df)

df_features = feature_model.transform(df)

In [0]:
display(
    df_features.select(
        "features"
    )
)

features
"{""type"":""0"",""size"":""29"",""indices"":[""0"",""1"",""3"",""5"",""6"",""9"",""13"",""19"",""24"",""26""],""values"":[""20.0"",""4.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0""]}"
"{""type"":""0"",""size"":""29"",""indices"":[""0"",""1"",""5"",""6"",""9"",""17"",""18"",""23"",""26""],""values"":[""24.0"",""3.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0""]}"
"{""type"":""0"",""size"":""29"",""indices"":[""0"",""1"",""4"",""6"",""9"",""17"",""19"",""23"",""26""],""values"":[""22.0"",""3.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0""]}"
"{""type"":""0"",""size"":""29"",""indices"":[""0"",""1"",""2"",""3"",""5"",""6"",""9"",""13"",""18"",""24"",""26""],""values"":[""22.0"",""6.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0""]}"
"{""type"":""0"",""size"":""29"",""indices"":[""0"",""1"",""3"",""4"",""6"",""9"",""17"",""19"",""24"",""26""],""values"":[""22.0"",""4.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0""]}"
"{""type"":""0"",""size"":""29"",""indices"":[""0"",""1"",""5"",""7"",""10"",""15"",""19"",""23"",""27""],""values"":[""27.0"",""2.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0""]}"
"{""type"":""0"",""size"":""29"",""indices"":[""0"",""1"",""4"",""6"",""9"",""13"",""18"",""23"",""26""],""values"":[""22.0"",""3.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0""]}"
"{""type"":""0"",""size"":""29"",""indices"":[""0"",""1"",""5"",""6"",""9"",""13"",""19"",""23"",""26""],""values"":[""24.0"",""3.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0""]}"
"{""type"":""0"",""size"":""29"",""indices"":[""0"",""1"",""5"",""6"",""9"",""13"",""19"",""23"",""26""],""values"":[""23.0"",""2.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0""]}"
"{""type"":""0"",""size"":""29"",""indices"":[""0"",""1"",""3"",""5"",""6"",""9"",""13"",""19"",""24"",""26""],""values"":[""23.0"",""4.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0""]}"


In [0]:
dataset_output = df_features.select(
    "features",
    "label_output"
)

In [0]:
dataset_feedback = df_features.select(
    "features",
    "label_feedback"
)

In [0]:
# Modelo de output para treinamento 80/20

train_feedback, test_feedback = (
    dataset_feedback.randomSplit(
        [0.8, 0.2],
        seed=42
    )
)

In [0]:
train_output, test_output = (
    dataset_output.randomSplit(
        [0.8, 0.2],
        seed=42
    )
)

print("Treino:", train_output.count())
print("Teste:", test_output.count())

Treino: 331
Teste: 57


In [0]:
#Treinamento do modelo de output
from pyspark.ml.classification import RandomForestClassifier

rf_output = RandomForestClassifier(
    labelCol="label_output",
    featuresCol="features",
    numTrees=300,
    maxDepth=8,
    seed=42
)

model_output = rf_output.fit(train_output)

In [0]:
# Treinamento do modelo de feedback
rf_feedback = RandomForestClassifier(
    labelCol="label_feedback",
    featuresCol="features",
    numTrees=300,
    maxDepth=8,
    seed=42
)

model_feedback = rf_feedback.fit(train_feedback)

In [0]:
# Fazer Previsões
pred_output = model_output.transform(test_output)

pred_feedback = model_feedback.transform(test_feedback)

In [0]:
# Calcular AUC para avaliar performance do modelo
from pyspark.ml.evaluation import BinaryClassificationEvaluator

evaluator = BinaryClassificationEvaluator(
    labelCol="label_output",
    metricName="areaUnderROC"
)

auc = evaluator.evaluate(pred_output)

print(f"AUC: {auc:.4f}")

AUC: 0.8056


In [0]:
importances = model_output.featureImportances
feature_names = assembler.getInputCols()

for f, imp in zip(feature_names, importances):
    print(f, imp)


age 0.1920528396420566
family_size 0.08908490809389362
big_family 0.022286667081230987
is_loyal 0.021494970897223974
gender_ohe 0.03383339440583732
marital_status_ohe 0.028023628276972423
occupation_ohe 0.04447307482439288
monthly_income_ohe 0.033386328839280686
educational_qualifications_ohe 0.017910377628411474
customer_type_ohe 0.03742837805258416
age_group_ohe 0.040393175755970526


In [0]:
# Ranking de influência dos atributos

importance_df = spark.createDataFrame(
    zip(feature_names,
        [float(x) for x in importances]),
    ["feature", "importance"]
)

display(
    importance_df.orderBy(
        importance_df.importance.desc()
    )
)

feature,importance
age,0.1920528396420566
family_size,0.08908490809389362
occupation_ohe,0.04447307482439288
age_group_ohe,0.040393175755970526
customer_type_ohe,0.03742837805258416
gender_ohe,0.03383339440583732
monthly_income_ohe,0.033386328839280686
marital_status_ohe,0.028023628276972423
big_family,0.022286667081230987
is_loyal,0.021494970897223974


**PROCESSO MLflow**

In [0]:
import mlflow
import mlflow.spark

In [0]:
# Nova avaliação do modelo

from pyspark.ml.evaluation import BinaryClassificationEvaluator

pred_output = model_output.transform(test_output)

evaluator = BinaryClassificationEvaluator(
    labelCol="label_output",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

auc = evaluator.evaluate(pred_output)

print(f"AUC = {auc}")

AUC = 0.8055555555555556


In [0]:
#%sql
#CREATE CATALOG main;
#CREATE SCHEMA main.delivery;
#CREATE VOLUME main.delivery.mlflow_volume;

In [0]:
mlflow.end_run()

In [0]:

import os

os.environ["MLFLOW_DFS_TMP"] = "/Volumes/main/delivery/mlflow_volume/tmp"

with mlflow.start_run(run_name="rf_output_model"):

    mlflow.log_param("numTrees", 300)
    mlflow.log_param("maxDepth", 8)

    mlflow.log_metric("AUC", auc)

    mlflow.spark.log_model(
        model_output,
        artifact_path="random_forest_output"
    )
    
    print("Modelo registrado com sucesso")

2026/09/21 23:36:09 WARNING mlflow.utils.requirements_utils: Found pyspark version (4.1.0+databricks.connect.18.1.9) contains a local version label (+databricks.connect.18.1.9). MLflow logged a pip requirement for this package as 'pyspark==4.1.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/09/21 23:36:14 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /local_disk0/user_tmp_data/spark-01858e03-e1a1-409c-87f6-87/tmpn28j86so/model, flavor: spark). Fall back to return ['pyspark==4.1.0']. Set logging level to DEBUG to see the full traceback. 
2026/09/21 23:36:14 INFO mlflow.models.model: Model logged without a signature. Signatures are required for Databricks UC model registry as they validate model inputs and denote the expected schema of model outputs. Please set `input_example` parameter when l

Modelo registrado com sucesso
